In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
FT_1ST_YEAR_SRC = pd.read_csv(r"S:\2026\01-Jan-26\01.01.26\28687976_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
FT_2ND_YEAR_SRC = pd.read_csv(r"S:\22.09.26\30956921_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
BEINDATA_SRC = pd.read_csv(r"S:\22.09.26\30956925_BEINDATANEWRPT.CSV", encoding='cp1256',dtype='str')

In [ ]:
FT_1ST_YEAR_SRC.loc[FT_1ST_YEAR_SRC['Subscriber Nr']=='19165740']

In [ ]:
first_year_from = '2025-01-01'
first_year_to = '2025-08-31'
second_year_from = '2026-01-01'
second_year_to = '2026-08-31'

In [ ]:
FT_1ST_YEAR_SRC['Created Date'] = pd.to_datetime(FT_1ST_YEAR_SRC['Created Date'], dayfirst=True)
FT_2ND_YEAR_SRC['Created Date'] = pd.to_datetime(FT_2ND_YEAR_SRC['Created Date'], dayfirst=True)
BEINDATA_SRC['Start Date'] = pd.to_datetime(BEINDATA_SRC['Start Date'], dayfirst=True, errors='coerce')

ft_2nd_year = FT_2ND_YEAR_SRC.copy()
ft_1st_year = FT_1ST_YEAR_SRC.copy()

beindata = BEINDATA_SRC.copy()

In [ ]:

# showrooms = [ "Maadi showroom","Mohandeseen Showroom" ]
dth_types = ['beIN Quartar Installment', 'CNE Subscriber', 'MCE staff (CNE staff)',
                'BeIN sports CC', 'beIN Bi Installment', 'Corporate Subscriber', 'Temp',
                'Bein NC', 'Bulk DTH customer', 'beIN Installment Sub', 'Charge Back']

plan_filter = (
            beindata["Plan"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            beindata["Plan"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            beindata["Plan"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )

invoice_types = ['Subscription Invoice']

Walk in subs in 
2025

In [ ]:
ft_1st_year_plan_filter = (
            ft_1st_year["Plan Name"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            ft_1st_year["Plan Name"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            ft_1st_year["Plan Name"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )

ft_1st_year = FT_1ST_YEAR_SRC.copy()
ft_1st_year = ft_1st_year.loc[ft_1st_year['Created Date'].between(pd.to_datetime(first_year_from),pd.to_datetime(first_year_to))]
ft_1st_year = ft_1st_year.loc[ft_1st_year['Default Entity Type'].isin(['CNE Dealer'])]
# ft_1st_year = ft_1st_year.loc[ft_1st_year['Subscriber Type'].isin(dth_types)]
ft_1st_year = ft_1st_year.loc[(ft_1st_year['Doc Type'] =='Invoice') & (ft_1st_year['Doc Status'] =='Posted')]
ft_1st_year = ft_1st_year.loc[ft_1st_year_plan_filter]

ft_1st_year = ft_1st_year.drop_duplicates(subset=['Subscriber Nr'])

ft_1st_year.to_csv('ft_1st_year check with doaa.csv', index=False)

print (f'Total contracts created in 2025: {ft_1st_year.shape[0]}')


In [ ]:
beindata = BEINDATA_SRC.copy()
beindata = beindata.loc[beindata['Start Date'].between(pd.to_datetime(first_year_from), pd.to_datetime(first_year_to))]

In [ ]:
ft_1st_year_bein = ft_1st_year.merge(right=beindata, left_on='Contract Number', right_on='Contract Number',how='inner').drop_duplicates(subset=['Subscriber Nr'])


In [ ]:
ft_1st_year_bein = ft_1st_year_bein.drop_duplicates(subset='Subscriber Nr')
ft_1st_year_bein.shape[0]

In [ ]:
disconnected_beindata = BEINDATA_SRC.copy()
active_beindata = BEINDATA_SRC.copy()

active_beindata = active_beindata.loc[(active_beindata['Status']=='Active') | (active_beindata['Status']=='Suspended')]
disconnected_beindata.loc[disconnected_beindata['Status']=='DIS']

disconnected_beindata = disconnected_beindata.loc[~disconnected_beindata['Customer Number'].isin(active_beindata['Customer Number'])]


active_beindata = active_beindata.drop_duplicates(subset=['Customer Number'])

active_beindata = active_beindata.loc[active_beindata['Customer Type'].isin(dth_types)]

print(disconnected_beindata.shape[0])

print(active_beindata.shape[0])


In [ ]:
disconnected_beindata.columns

In [ ]:
ft_1st_year_bein_disconnected = ft_1st_year_bein.merge(right=disconnected_beindata[['Customer Number','Status','Entity']], left_on='Subscriber Nr', right_on='Customer Number', how='inner')
ft_1st_year_bein_disconnected = ft_1st_year_bein_disconnected.drop_duplicates(subset=['Subscriber Nr'])
ft_1st_year_bein_disconnected.shape[0]

In [ ]:
ft_1st_year_bein_active = ft_1st_year_bein.merge(right=active_beindata[['Customer Number','Status','Entity']], left_on='Subscriber Nr', right_on='Customer Number', how='inner')
ft_1st_year_bein_active = ft_1st_year_bein_active.drop_duplicates(subset=['Subscriber Nr'])
ft_1st_year_bein_active.shape[0]

In [ ]:
pd.concat([ft_1st_year_bein_disconnected,ft_1st_year_bein_active]).to_csv('final_ft_cne_dealers.csv', index=False)